In [2]:
import simnibs
from simnibs import mesh_io
import numpy as np
import nibabel as nib
import pandas as pd
import os

In [7]:


def crop_mesh_with_node_mask(msh, node_mask):
    """
    Crop a mesh using a boolean node mask.
    
    Parameters:
    msh: simnibs.msh.Msh
        The original mesh object
    node_mask: numpy array (bool)
        Boolean array where True indicates nodes to keep
        
    Returns:
    simnibs.msh.Msh
        Cropped mesh containing only elements with at least one selected node
    """
    # Convert the mask to node indices (1-based)
    selected_node_indices = np.where(node_mask)[0] + 1  # +1 for 1-based indexing
    
    # Find elements that contain at least one selected node
    elements_to_keep = []
    for el in msh.elm:
        # Check if any node in this element is selected
        if any(node in selected_node_indices for node in el.node_number_list):
            elements_to_keep.append(True)
        else:
            elements_to_keep.append(False)
    
    # Get all nodes that are in the kept elements
    all_nodes_in_elements = set()
    for el in msh.elm[elements_to_keep]:
        all_nodes_in_elements.update(el.node_number_list)
    
    # Create new node and element structures
    new_nodes = msh.nodes[list(all_nodes_in_elements)]
    new_elements = msh.elm[elements_to_keep]
    
    # Create a new mesh with the cropped data
    cropped_msh = simnibs.Msh(nodes=new_nodes, elements=new_elements)
    
    return cropped_msh

In [17]:
mesh_path = "E:/Datasets/Stage 2/Meshes/m2m_TCGA-02-0006/TCGA-02-0006.msh"
result_mesh = mesh_io.read_msh(mesh_path)

scalp_surface = result_mesh.crop_mesh(1005)
nodes = scalp_surface.nodes.node_coord


# Get min/max for Z (height) and Y (depth)
z_min, z_max = np.min(nodes[:, 2]), np.max(nodes[:, 2])  # Z: inferior-superior
y_min, y_max = np.min(nodes[:, 1]), np.max(nodes[:, 1])  # Y: posterior-anterior

# Define thresholds as percentages (e.g., 30% from bottom, 20% from front)
z_threshold = z_min + 0.55 * (z_max - z_min)  # Keep nodes above 45% of head height
y_threshold = y_min + 0.20 * (y_max - y_min)  # Keep nodes behind 20% of head depth
z2_threshold = z_min + 35

# Combined condition: above eyebrows OR behind ears and with normal not directed into -z
mask = ((nodes[:, 2] > z_threshold) | ((nodes[:, 1] < y_threshold) & (nodes[:, 2] > z2_threshold)))
# Filter nodes and normals
filtered_nodes = scalp_surface.nodes[mask]
selected_node_indices = np.where(mask)[0] + 1
croped_mesh = scalp_surface.crop_mesh(nodes = selected_node_indices)

#Electrode = result_mesh.crop_mesh(3400)
v = croped_mesh.view()
v.show()